# Human/GPT cross-persistence density analysis

Supports Appendix E: qualitative comparison of human-written and AI-generated texts.

Run this notebook from the repository root. Large datasets and generated intermediate artifacts are not committed; see `README.md` for the expected layout.


In [ ]:
import numpy as np

import mtd

from tqdm import tqdm

In [ ]:
import pickle

with open("Data/cross_ripsnet_text_exp/human_gpt3_davinci_003_pc_train", "rb") as fp:   #Pickling
    pc_train = pickle.load(fp)

with open("Data/cross_ripsnet_text_exp/human_gpt3_davinci_003_train_indexes", "rb") as fp:   #Pickling
    train_indexes = pickle.load(fp)

with open("Data/cross_ripsnet_text_exp/human_gpt3_davinci_003_train_indexes", "rb") as fp:   #Pickling
    train_indexes = pickle.load(fp)


cloud_dim = pc_train[0].shape[-1]
mid = 2000
end = 2500

In [ ]:
train_PD = []
for i,j in tqdm(train_indexes):  
    batch_size1 = int(len(pc_train[i]))
    batch_size2 = int(len(pc_train[i]))
    barc = [mtd.calc_cross_barcodes(pc_train[i], pc_train[i], batch_size1 = batch_size1, batch_size2 = 0, is_plot = False, pdist_device = "cuda"),
            mtd.calc_cross_barcodes(pc_train[j], pc_train[j], batch_size1 = batch_size2, batch_size2 = 0, is_plot = False, pdist_device = "cuda")]
    train_PD.append(barc)

In [ ]:
with open("Data/cross_ripsnet_text_exp/human_gpt3_pds_pds_for_visualization", "wb") as fp:   #Pickling
    pickle.dump(train_PD, fp)

In [ ]:
train_PD_1 = [[x[1] if len(x[1])>0 else np.array([[0., 0.]]) for x in barcs ] for barcs in train_PD]

In [ ]:
from gudhi.representations import DiagramSelector
pds_train = DiagramSelector(use=True).fit_transform(train_PD_1[0])
vpdtr = np.vstack(pds_train)

for barcs in tqdm(train_PD_1[1:]):
    pds_train = DiagramSelector(use=True).fit_transform(barcs)
    vpdtr = np.vstack((vpdtr, np.vstack(pds_train)))

pers = vpdtr[:,1]-vpdtr[:,0]
im_bnds = [np.min(vpdtr[:,0]), np.max(vpdtr[:,0]), np.min(pers), np.max(pers)]

In [ ]:
from sklearn.metrics import pairwise_distances
from gudhi.representations import  PersistenceImage

PI_train_all = []
for barcs in tqdm(train_PD_1):
    pds_train = DiagramSelector(use=True).fit_transform(barcs)
    pds_train = [item.astype(np.float32) for item in pds_train]
    # clean_pds_test = DiagramSelector(use=True).fit_transform(train_PD_1[800:])
    
    
    vpdtr = np.vstack(pds_train)
    pers = vpdtr[:,1]-vpdtr[:,0]
    bps_pairs = pairwise_distances(np.hstack([vpdtr[:,0:1],vpdtr[:,1:2]-vpdtr[:,0:1]])[:200]).flatten()
    ppers = bps_pairs[np.argwhere(bps_pairs > 1e-5).ravel()]
    sigma = np.quantile(ppers, .01)
    
    
    PI_params = {'bandwidth': sigma, 'weight': lambda x: x[1]**2, 
                 'resolution': [50,50], 'im_range': im_bnds}
    PI_train = PersistenceImage(**PI_params).fit_transform(pds_train)
    # clean_PI_test = PersistenceImage(**PI_params).fit_transform(clean_pds_test)
    PI_train = PI_train/np.sum(PI_train, axis = 1)[:, np.newaxis]
    PI_train_all.append(PI_train)
    
PI_train_all = np.array(PI_train_all)

In [ ]:
fig = plt.figure(figsize=(8, 120)) 
gs = gridspec.GridSpec(40, 2, width_ratios=[1,1], wspace=2.0, hspace=0.0)
for i in range(40):
    idx = i
    ax = plt.subplot(gs[i, 0])
    ax.imshow(np.flip(np.reshape(PI_train_all[idx][0], [50,50]), 0), cmap='jet')
    plt.xticks([])
    plt.yticks([])
    
    ax = plt.subplot(gs[i, 1])
    ax.imshow(np.flip(np.reshape(PI_train_all[idx][1], [50,50]), 0), cmap='jet')
    plt.xticks([])
    plt.yticks([])

In [ ]:
PI_train_all.shape

In [ ]:
PI_train_all_vec = PI_train_all.reshape(10000, -1)

In [ ]:
PI_train_all_vec.shape

In [ ]:
# PI_pictures = [np.flip(np.reshape(pic, [50,50]), 0) for pic in PI_train_all]
classes = {"1":"GPT", "0": "Human"}

In [ ]:
labels_of_left_clouds = [classes[str(train_indexes[idx][0]%2)] for idx in range(len(train_indexes))]

In [ ]:
import plotly.graph_objs as gobj

def plot_point_cloud_my(point_cloud, labels, dimension=None, plotly_params=None):
    # TODO: increase the marker size
    if dimension is None:
        dimension = np.min((3, point_cloud.shape[1]))

    # Check consistency between point_cloud and dimension
    if point_cloud.shape[1] < dimension:
        raise ValueError("Not enough dimensions available in the input point "
                         "cloud.")

    elif dimension == 2:
        layout = {
            "width": 600,
            "height": 600,
            "xaxis1": {
                "title": "0th",
                "side": "bottom",
                "type": "linear",
                "ticks": "outside",
                "anchor": "x1",
                "showline": True,
                "zeroline": True,
                "showexponent": "all",
                "exponentformat": "e"
                },
            "yaxis1": {
                "title": "1st",
                "side": "left",
                "type": "linear",
                "ticks": "outside",
                "anchor": "y1",
                "showline": True,
                "zeroline": True,
                "showexponent": "all",
                "exponentformat": "e"
                },
            "plot_bgcolor": "white"
            }

        fig = gobj.Figure(layout=layout)
        fig.update_xaxes(zeroline=True, linewidth=1, linecolor="black",
                         mirror=False)
        fig.update_yaxes(zeroline=True, linewidth=1, linecolor="black",
                         mirror=False)

        fig.add_trace(gobj.Scatter(
            x=point_cloud[:, 0],
            y=point_cloud[:, 1],
            mode="markers",
            marker={"size": 4,
                    "color": colors,
                    "colorscale": "Viridis",
                    "opacity": 0.8}
            ))

    elif dimension == 3:
        scene = {
            "xaxis": {
                "title": "0th",
                "type": "linear",
                "showexponent": "all",
                "exponentformat": "e"
                },
            "yaxis": {
                "title": "1st",
                "type": "linear",
                "showexponent": "all",
                "exponentformat": "e"
                },
            "zaxis": {
                "title": "2nd",
                "type": "linear",
                "showexponent": "all",
                "exponentformat": "e"
                }
            }

        fig = gobj.Figure()
        fig.update_layout(scene=scene)
        for tag in ["GPT", "Human"]:
            mask = [g == tag for g in labels]
            fig.add_trace(gobj.Scatter3d(
                x=point_cloud[mask, 0],
                y=point_cloud[mask, 1],
                z=point_cloud[mask, 2],
                mode="markers",
                name = tag,
                marker={"size": 4,
                        # "color": list(range(point_cloud.shape[0])),
                        "color": "orange" if tag=="GPT" else "blue",
                        "colorscale": "Viridis",
                        "opacity": 0.8}
                ))
    fig.update_layout(showlegend=True) 
    # Update trace and layout according to user input
    if plotly_params:
        fig.update_traces(plotly_params.get("trace", None))
        fig.update_layout(plotly_params.get("layout", None))

    return fig

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.manifold import Isomap
from sklearn.manifold import LocallyLinearEmbedding


pca = PCA(n_components=3)
two_d_pics_pca = pca.fit_transform(PI_train_all_vec)

print("PCA done")

isomap = Isomap(n_components=3)
two_d_pics_isomap = isomap.fit_transform(PI_train_all_vec)

print("Isomap done")

tsne = TSNE(n_components=3)
two_d_pics_tsne = tsne.fit_transform(PI_train_all_vec)

print("TSNE done")

lle = LocallyLinearEmbedding(n_components=3)
two_d_pics_lle = lle.fit_transform(PI_train_all_vec)

print("LocallyLinearEmbedding done")

In [ ]:
plot_point_cloud_my(two_d_pics_tsne, labels = labels_of_left_clouds)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler

def plot_dimreduction_result(data, labels, all_pictures, title, plot_pics = True):
    X = MinMaxScaler().fit_transform(data)
    _, ax = plt.subplots()
    sns.scatterplot(x=X[:,0], y=X[:,1], hue=labels, legend=True, ax = ax,alpha = 0.7)
    if plot_pics:
        shown_images = np.array([[1.0, 1.0]])  # just something big
        for i in range(X.shape[0]):
            # plot every digit on the embedding
            # show an annotation box for a group of digits
            dist = np.sum((X[i] - shown_images) ** 2, 1)
            if np.min(dist) < 4e-3:
                # don't show points that are too close
                continue
            shown_images = np.concatenate([shown_images, [X[i]]], axis=0)
            imagebox = offsetbox.AnnotationBbox(
                offsetbox.OffsetImage(all_pictures[i],zoom = 0.3, cmap='jet'), X[i], pad = 0.1
            )
            imagebox.set(zorder=1)
            ax.add_artist(imagebox)

    ax.set_title(title)
    ax.axis("off")
    # plt.scatter(two_d_pics_isomap [:,0], two_d_pics_isomap [:,1],)
    # plt.title("Isomap")
    plt.show()


plot_dimreduction_result(two_d_pics_pca, labels_of_left_clouds, [], title = "PCA", plot_pics=False)

plot_dimreduction_result(two_d_pics_isomap, labels_of_left_clouds, [], title = "Isomap", plot_pics=False)

plot_dimreduction_result(two_d_pics_tsne, labels_of_left_clouds, [], title = "TSNE", plot_pics=False)

plot_dimreduction_result(two_d_pics_lle, labels_of_left_clouds, [], title = "LocallyLinearEmbedding", plot_pics=False)